In [ ]:
import mario
import yaml
import pandas as pd
import os

import warnings
warnings.filterwarnings("ignore")

user = 'LR'   # change this to your username
years = range(2024,2026)   
versions = ['v1.0', 'v2.0']  # versions to parse

with open('paths.yml', 'r') as file: # open the yml file
    paths = yaml.safe_load(file)

paths = paths[user]

In [ ]:
footprints = pd.DataFrame()
ghgs = [
    "Carbon dioxide, fossil (air - Emiss)",
    "CH4 (air - Emiss)",
    "N2O (air - Emiss)",
]

for version in versions:
    for year in years:
        db = mario.parse_from_txt(
            path = os.path.join(paths['export'], version, str(year)),
            mode = "flows",
            table = 'SUT',
        )

        f = db.f.loc[ghgs,:]
        f = f.T
        f['GHG'] = f['Carbon dioxide, fossil (air - Emiss)'] + \
                   f['CH4 (air - Emiss)']*25 + \
                   f['N2O (air - Emiss)']*298

        f.columns.names = ['Substances']
        f = f.stack().to_frame()
        f.columns = ['Value']
        f.reset_index(inplace=True)
        f['Year'] = year
        f['Version'] = version

        footprints = pd.concat([footprints, f], axis=0, ignore_index=True)

In [ ]:
footprints.to_csv(
    paths['export']+"_results/Footprints.csv"
)